## Data Pre-processing

In [1]:
from litellm import completion
from dotenv import load_dotenv
import json
from pricer.batch import Batch
from pricer.items import Item

load_dotenv(override=True)

True

In [2]:
LITE_MODE = True

In [3]:
username = "SeanSunny"
dataset = f"{username}/items_raw_lite" if LITE_MODE else f"{username}/items_raw_full"

train, val, test = Item.from_hub(dataset)

items = train + val + test

print(f"Loaded {len(items):,} items")
print(items[0])

Loaded 22,000 items
title='Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only)' category='Tools_and_Home_Improvement' price=64.3 full='Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only)\n[\'From the Manufacturer\', "When you have a Schlage handleset on your front door, you ensure your security as well as your peace of mind. After all, we\'re the leader in security devices, trusted for over 85 years. All Schlage handlesets are precision engineered, featuring 100% solid"]\n[\'Interior half only\', \'Requires F58 to complete handle set\', \'Non handed knob style\', \'4" minimum center to center door prep required for this two piece model.\', \'Lifetime Mechanical and Finish Warranty\']\n{"Material": "Metal", "Brand": "", "Color": "Oil Rubbed Bronze", "Exterior Finish": "Bronze", "Special Feature": "Easy to Install", "Age Range (Description)": "Adult", "Included Components": "Deadbolt, Knob", "Item Wei

In [4]:
print(items[0])

title='Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only)' category='Tools_and_Home_Improvement' price=64.3 full='Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only)\n[\'From the Manufacturer\', "When you have a Schlage handleset on your front door, you ensure your security as well as your peace of mind. After all, we\'re the leader in security devices, trusted for over 85 years. All Schlage handlesets are precision engineered, featuring 100% solid"]\n[\'Interior half only\', \'Requires F58 to complete handle set\', \'Non handed knob style\', \'4" minimum center to center door prep required for this two piece model.\', \'Lifetime Mechanical and Finish Warranty\']\n{"Material": "Metal", "Brand": "", "Color": "Oil Rubbed Bronze", "Exterior Finish": "Bronze", "Special Feature": "Easy to Install", "Age Range (Description)": "Adult", "Included Components": "Deadbolt, Knob", "Item Weight": "1.5 pounds", 

In [5]:
items[0]

<Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only) = $64.3>

In [6]:
items[0].title

'Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only)'

In [9]:
items[0].category

'Tools_and_Home_Improvement'

In [10]:
items[0].price

64.3

In [7]:
items[0].full

'Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only)\n[\'From the Manufacturer\', "When you have a Schlage handleset on your front door, you ensure your security as well as your peace of mind. After all, we\'re the leader in security devices, trusted for over 85 years. All Schlage handlesets are precision engineered, featuring 100% solid"]\n[\'Interior half only\', \'Requires F58 to complete handle set\', \'Non handed knob style\', \'4" minimum center to center door prep required for this two piece model.\', \'Lifetime Mechanical and Finish Warranty\']\n{"Material": "Metal", "Brand": "", "Color": "Oil Rubbed Bronze", "Exterior Finish": "Bronze", "Special Feature": "Easy to Install", "Age Range (Description)": "Adult", "Included Components": "Deadbolt, Knob", "Item Weight": "1.5 pounds", "Handle Material": "Bronze", "Package Type": "Standard Packaging", "Unit Count": "1.0 Count", "Number of Items": "1", "Manufacturer": "Schlage", "Product Dime

In [15]:
items[0].weight

1.5

In [8]:
items[0].id

In [9]:
# Give every item an id

for index, item in enumerate(items):
    item.id = index

In [10]:
items[0].id

0

In [11]:
SYSTEM_PROMPT = """Create a concise description of a product. Respond only in this format. Do not include part numbers.
Title: Rewritten short precise title
Category: eg Electronics
Brand: Brand name
Description: 1 sentence description
Details: 1 sentence on features"""

In [12]:
print(items[0].full)

Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only)
['From the Manufacturer', "When you have a Schlage handleset on your front door, you ensure your security as well as your peace of mind. After all, we're the leader in security devices, trusted for over 85 years. All Schlage handlesets are precision engineered, featuring 100% solid"]
['Interior half only', 'Requires F58 to complete handle set', 'Non handed knob style', '4" minimum center to center door prep required for this two piece model.', 'Lifetime Mechanical and Finish Warranty']
{"Material": "Metal", "Brand": "", "Color": "Oil Rubbed Bronze", "Exterior Finish": "Bronze", "Special Feature": "Easy to Install", "Age Range (Description)": "Adult", "Included Components": "Deadbolt, Knob", "Item Weight": "1.5 pounds", "Handle Material": "Bronze", "Package Type": "Standard Packaging", "Unit Count": "1.0 Count", "Number of Items": "1", "Manufacturer": "Schlage", "Product Dimensions": "8.1 x 4

In [13]:
messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": items[0].full}]
response = completion(messages=messages, model="groq/openai/gpt-oss-20b", reasoning_effort="low")

print(response.choices[0].message.content)
print()
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Cost: {response._hidden_params['response_cost']*100:.3f} cents")


Title: Schlage F59 Interior Half Knob with Deadbolt – Oil Rubbed Bronze  
Category: Hardware  
Brand: Schlage  
Description: A precision‑engineered interior half knob with integrated deadbolt, finished in oil‑rubbed bronze for durable style and security.  
Details: Features easy installation, a 4‑inch minimum center‑to‑center door prep, and a lifetime mechanical and finish warranty.

Input tokens: 446
Output tokens: 122
Cost: 0.007 cents


In [14]:
MODEL = "openai/gpt-oss-20b"


In [15]:
def make_jsonl(item):
    body = {"model": MODEL, "messages": [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": item.full}], "reasoning_effort": "low"}
    line = {"custom_id": str(item.id), "method": "POST", "url": "/v1/chat/completions", "body": body}
    return json.dumps(line)

In [16]:
items[0]

<Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only) = $64.3>

In [17]:
make_jsonl(items[0])

'{"custom_id": "0", "method": "POST", "url": "/v1/chat/completions", "body": {"model": "openai/gpt-oss-20b", "messages": [{"role": "system", "content": "Create a concise description of a product. Respond only in this format. Do not include part numbers.\\nTitle: Rewritten short precise title\\nCategory: eg Electronics\\nBrand: Brand name\\nDescription: 1 sentence description\\nDetails: 1 sentence on features"}, {"role": "user", "content": "Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only)\\n[\'From the Manufacturer\', \\"When you have a Schlage handleset on your front door, you ensure your security as well as your peace of mind. After all, we\'re the leader in security devices, trusted for over 85 years. All Schlage handlesets are precision engineered, featuring 100% solid\\"]\\n[\'Interior half only\', \'Requires F58 to complete handle set\', \'Non handed knob style\', \'4\\" minimum center to center door prep required for this two piece m

In [18]:

def make_file(start, end, filename):
    batch_file = filename
    with open(batch_file, "w", encoding="utf-8") as f:
        for i in range(start, end):
            f.write(make_jsonl(items[i]))
            f.write("\n")

In [19]:
make_file(0, 10, "jsonl/0_10.jsonl")

In [20]:
import os
from groq import Groq

groq = Groq(api_key=os.environ.get("GROQ_API_KEY"))

In [22]:
with open("jsonl/0_10.jsonl", "rb") as f:
    response = groq.files.create(file=f, purpose="batch")
response

FileCreateResponse(id='file_01kr5958vyedfsvm41x2txsmem', bytes=23695, created_at=1778293842, filename='0_10.jsonl', object='file', purpose='batch', size=0, md5='XQCSuW0y+CPhuhOc0Qn3qA==', content_type='application/jsonl')

In [23]:
file_id = response.id
file_id

'file_01kr5958vyedfsvm41x2txsmem'

In [24]:
response = groq.batches.create(completion_window="24h", endpoint="/v1/chat/completions", input_file_id=file_id)
response

BatchCreateResponse(id='batch_01kr595pw7fwgt4k15ce3jyxcj', completion_window='24h', created_at=1778293857, endpoint='/v1/chat/completions', input_file_id='file_01kr5958vyedfsvm41x2txsmem', object='batch', status='validating', cancelled_at=None, cancelling_at=None, completed_at=None, error_file_id=None, errors=None, expired_at=None, expires_at=1778380257, failed_at=None, finalizing_at=None, in_progress_at=None, metadata=None, output_file_id=None, request_counts=RequestCounts(completed=0, failed=0, total=0), project_id='project_01jyvhwdcxfg8byqkxzt94knvn')

In [27]:
result = groq.batches.retrieve(response.id)
result

BatchRetrieveResponse(id='batch_01kr595pw7fwgt4k15ce3jyxcj', completion_window='24h', created_at=1778293857, endpoint='/v1/chat/completions', input_file_id='file_01kr5958vyedfsvm41x2txsmem', object='batch', status='completed', cancelled_at=None, cancelling_at=None, completed_at=1778293860, error_file_id=None, errors=None, expired_at=None, expires_at=1778380257, failed_at=None, finalizing_at=1778293859, in_progress_at=1778293859, metadata=None, output_file_id='file_01kr595smre7jvbpfpjen4jx8n', request_counts=RequestCounts(completed=10, failed=0, total=10), project_id='project_01jyvhwdcxfg8byqkxzt94knvn')

In [28]:
response = groq.files.content(result.output_file_id)
response.write_to_file("jsonl/batch_results.jsonl")

In [29]:
with open("jsonl/batch_results.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        json_line = json.loads(line)
        id = int(json_line["custom_id"])
        summary = json_line["response"]["body"]["choices"][0]["message"]["content"]
        items[id].summary = summary


In [34]:
print(items[3].full)

Hollyland Mars 300 Pro Enhanced Wireless Video Transmitter & Receiver 300Ft Range 1080p HDMI 5G 0.08S Low Latency APP Support iOS Android [2 Battery Pack & AC Charger Bundle]
['KEY ', 'Built-In Antennas - Built-In Antennas for Easy Setups, Efficient Shooting, and Different Shooting Applications also Includes External Antennas300ft Transmission Range with 0.08s Minimum Latency - Up to 300ft Hassle-Free and Reliable Range for Wireless Video and Audio Transmission. 0.08S Lowest Achievable Latency for Real-Time MonitoringThumbwheel Switch Menu - Better Using Experience with the Multi-Functional Clickable Thumbwheel SwitchMultiple Power Options - Supports 5-12V Wide Voltage Power Supply, Including Various L-Series Batteries, Different Power Banks, and Type-C (5V/2A) ChargingHDMI In & Loopout - HDMI Input and loopout on TX, and Dual HDMI Outputs on RXSide OLED - Side OLED Provides Easy Access to Power Status, Channel Scan, and Other OLED Display Information for Both Vertical or Horizontal In

In [33]:
print(items[3].title)

Hollyland Mars 300 Pro Enhanced Wireless Video Transmitter & Receiver 300Ft Range 1080p HDMI 5G 0.08S Low Latency APP Support iOS Android [2 Battery Pack & AC Charger Bundle]


In [35]:
print(items[3].category)

Electronics


In [36]:
print(items[3].summary)

Title: Hollyland Mars 300 Pro 1080p Wireless Video Transmitter & Receiver  
Category: Electronics  
Brand: Hollyland  
Description: 1080p HDMI transmitter and receiver with 300 ft range, 0.08 s latency, and dual HDMI outputs.  
Details: Built‑in antennas, thumbwheel menu, OLED display, app monitoring, and included battery pack and AC charger.
